In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys

sys.path.insert(0, "../..")

In [2]:
import chex
import dill
import jax
import jax.numpy as jnp
import json
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scipy
import torch
import treescope

from flax import nnx
from torch.utils.data import DataLoader
from typing import Any, NamedTuple
from typing_extensions import Protocol, runtime_checkable

from src.dataset import get_iter
from src.datasets.repetition import Repetition
from src.decoding import make_autoregressive
from src.rollout import rollout
from src.utils import parse_dict
from src.verifier import make_compute_returns

# Penzai
from penzai import pz

import IPython

pz.ts.register_as_default()

# Optional automatic array visualization extras:
pz.ts.register_autovisualize_magic()
pz.enable_interactive_context()
pz.ts.active_autovisualizer.set_interactive(pz.ts.ArrayAutovisualizer())


In [3]:
base_path = "/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/results"
algo_name = "debug"
# run_name = "reinforce-explore_only-in_traj-scratch_1M-0_cot_tokens-8x8-02-12-26_13_06_55-675fea0d-1fdf-4143-8bbe-2d210bc4bf39"
# run_name = "reinforce-explore_only-large_act_space-in_traj-scratch_1M-0_cot_tokens-8x8-02-12-26_13_31_37-bf844d3a-2297-40b6-8542-f15f1ed60f89"
# run_name = "reinforce-explore_only-large_act_space-quiet_softmax-in_traj-scratch_1M-0_cot_tokens-8x8-02-12-26_13_37_14-7b215eaa-6ee9-46b5-a785-50c7640a0571"
run_name = "reinforce-explore_only-large_act_space-smaller_model-in_traj-scratch_1M-0_cot_tokens-8x8-02-12-26_16_14_09-859a629e-a8bb-4c4c-8917-216e4cdd492f"
# algo_name = "debug_ntp"
# run_name = "ntp-quiet_softmax-scratch_1M-02-11-26_09_22_07-6b95fd18-3c76-44e4-9638-050d38f9e23d"

learner_path = os.path.join(base_path, algo_name, run_name)

In [4]:
eval_seed = 42
num_evals = 1
checkpoint_i = -1

In [5]:
config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))
half_precision = config_dict["half_precision"]
dtype = jnp.bfloat16 if half_precision else jnp.float32

# Get dataset
config = parse_dict(config_dict)
dataset_kwargs = config.dataset_kwargs
batch_size = dataset_kwargs.vocab_size

In [6]:
num_checkpoints = len(os.listdir(os.path.join(learner_path, "models")))

dataset = Repetition(
    dataset_kwargs.context_len,
    dataset_kwargs.vocab_size,
    dataset_kwargs.k,
    True,
    eval_seed,
    "question_only",
    1.0,
    getattr(dataset_kwargs, "num_repeats", None),
    False,
    True,
    0,
)

EOS TOKEN: 11
TOKEN MAP: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}


In [7]:
out_dim = int(dataset.output_space.n)
data_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
)
data_iter = get_iter(data_loader, None, dtype)

In [8]:
last_step = sorted(os.listdir(os.path.join(learner_path, "models")))[checkpoint_i]
train_state = dill.load(
    open(os.path.join(learner_path, "models", last_step), "rb")
)

model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)
model.set_attributes(deterministic=False, decode=False)

In [9]:
batch = next(data_iter)
batch = {
    k: np.repeat(v, num_evals, axis=0)
    for k, v in batch.items()
}

In [10]:
batch

{'sequence': array([[ 8, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 1, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 5, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 0, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 7, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 2, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 9, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 4, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 3, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [ 6, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11]]),
 'target': array([[10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11],
        [10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         11, 11, 11, 11, 11, 11]]),
 'mask': array([[False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True]]),
 'po

In [11]:
model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)

rng = jax.random.PRNGKey(eval_seed)
rng, rollout_rng = jax.random.split(rng)

# Sample next batch and evaluate
_, init_cache = make_autoregressive(
    model,
    max_decode_len=dataset_kwargs.context_len,
    batch_size=batch_size,
    embed_dim=config_dict["model_config"]["model_kwargs"]["embed_dim"],
    dtype=dtype,
    eval_mode=True,
)
cache = init_cache()
graphdef, _, rest = nnx.split(model, nnx.Cache, ...)

rollout_res = rollout(
    graphdef,
    cache,
    rest,
    rollout_rng,
    batch,
    eos_token=dataset.eos_token_id,
    attempt_length=config.attempt_length,
    deterministic=int(num_evals == 1),
    correct_aware_shift=dataset.correctness_aware_tokens_offset,
    max_token_id_to_shift=dataset.max_token_id_to_shift,
)

/home/bryanpu1/.conda/envs/parallel_vs_serial/lib/python3.10/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [12]:
model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)
model.set_attributes(deterministic=False, decode=False)

res = model({
    "sequence": rollout_res.observations
})

In [13]:
intermediates = nnx.pop(model, nnx.Intermediate)

In [14]:
rollout_res.success

Array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True], dtype=bool)

In [15]:
(batch["question_len"] + rollout_res.response_length).astype(int).shape

(10,)

In [16]:
res.shape, rollout_res.observations.shape

((10, 21, 10), (10, 21))

In [17]:
reset_idxes = np.where(rollout_res.observations[0] == dataset.reset_token_id)[0]
if rollout_res.success[0].item():
    reset_idxes = np.concatenate((reset_idxes, [np.argmax(rollout_res.observations[0] == dataset.eos_token_id)]))
reset_idxes[1:] - reset_idxes[:-1]

array([  2,   2,   2,   2,   2,   2,   2,   2,   2, -19])

In [18]:
treescope.render_array(
    rollout_res.observations[:, ::2]
)


<Arrayviz rendering>

In [19]:
for curr_obss in rollout_res.observations[:, 2::2]: 
    print(np.unique(curr_obss, return_counts=True))

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))


In [20]:
attn_weights = []
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(np.mean(
        intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0],
        axis=1,
    ))

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "sample", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"],
    # sliders=["sample"],
)

<Arrayviz rendering>

In [25]:
attn_weights = []
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(
        intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0]
    )

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "sample", "head", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"],
    sliders=["sample"],
)

<Arrayviz rendering>

In [24]:
train_state.params["embedders"]["token_emb"]["embedding"].value @ train_state.params["embedders"]["token_emb"]["embedding"].value.T

Array([[ 1.251944  ,  0.01402996,  0.28727782, -0.03976466, -0.17851664,
         0.49789894,  0.26215822, -0.4718924 , -0.12007321, -0.13435349,
        -0.34466752,  0.07856924],
       [ 0.01402996,  3.1010435 ,  0.3166328 , -0.13009477,  0.5500642 ,
         0.31767678,  0.37236708,  0.19724022, -0.79321027, -0.67113787,
        -0.34062687, -0.29512918],
       [ 0.28727782,  0.3166328 ,  1.2737818 , -0.24422939,  0.17441738,
         0.03128996,  0.4110621 , -0.13776526,  0.01316112,  0.02341893,
        -0.43106446, -0.02872767],
       [-0.03976466, -0.13009477, -0.24422939,  0.9682243 ,  0.50995666,
        -0.5812219 , -0.09906751, -0.05579863, -0.02913958,  0.01341663,
         0.1854756 , -0.3759202 ],
       [-0.17851664,  0.5500642 ,  0.17441738,  0.50995666,  3.0478451 ,
         0.24174264,  0.19987403, -0.58887225, -0.58591956, -0.20326906,
        -0.41622156, -0.3921777 ],
       [ 0.49789894,  0.31767678,  0.03128996, -0.5812219 ,  0.24174264,
         2.4703493 , -0.28837317, -0.7526315 , -0.41626024, -0.5788379 ,
        -0.1427144 ,  0.4031247 ],
       [ 0.26215822,  0.37236708,  0.4110621 , -0.09906751,  0.19987403,
        -0.28837317,  2.914924  , -1.2573398 , -0.53875417, -0.86102575,
        -0.32217905,  0.06672995],
       [-0.4718924 ,  0.19724022, -0.13776526, -0.05579863, -0.58887225,
        -0.7526315 , -1.2573398 ,  1.6709498 ,  0.2279361 ,  0.40828222,
        -0.12434258,  0.09309571],
       [-0.12007321, -0.79321027,  0.01316112, -0.02913958, -0.58591956,
        -0.41626024, -0.53875417,  0.2279361 ,  1.0762411 ,  0.5238193 ,
         0.18653964, -0.3193396 ],
       [-0.13435349, -0.67113787,  0.02341893,  0.01341663, -0.20326906,
        -0.5788379 , -0.86102575,  0.40828222,  0.5238193 ,  0.9241166 ,
         0.26211616,  0.07866693],
       [-0.34466752, -0.34062687, -0.43106446,  0.1854756 , -0.41622156,
        -0.1427144 , -0.32217905, -0.12434258,  0.18653964,  0.26211616,
         0.8069656 , -0.18011178],
       [ 0.07856924, -0.29512918, -0.02872767, -0.3759202 , -0.3921777 ,
         0.4031247 ,  0.06672995,  0.09309571, -0.3193396 ,  0.07866693,
        -0.18011178,  0.99480796]], dtype=float32)